# Lekcia 12 - Skrátenie histórie chatu pomocou agentového zápisníka

Táto poznámková kniha demonštruje, ako spravovať kontext v dlhých konverzáciách pomocou Microsoft Agent Framework. Ako konverzácie rastú, počet tokenov sa zvyšuje — nakoniec prekročí kontextové okno modelu. Toto riešime pomocou **vzoru súhrnu kontextu** a **agentového zápisníka** pre trvalú pamäť.

## Čo sa naučíte:
1. **Prečo je správa kontextu dôležitá**: Pochopenie limitov tokenov a kontextových okien
2. **Agentov citlivých na kontext**: Budovanie agentov, ktorí spravujú vlastný kontext konverzácie
3. **Vzor súhrnu kontextu**: Použitie nástrojov na zhrnutie histórie konverzácie
4. **Agentový zápisník**: Trvalá pamäť, ktorá pretrváva aj po znížení kontextu

## Predpoklady:
- Nastavenie Azure OpenAI s nakonfigurovanými premennými prostredia
- Pochopenie základných konceptov agentov z predchádzajúcich lekcií


## Nastavenie


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## Prečo je dôležité spravovanie kontextu

Každý LLM má obmedzené **kontextové okno** — maximálny počet tokenov, ktoré môže spracovať v jednom požiadavku. Ako sa konverzácia s viacerými kolami vyvíja:

- **Počet tokenov rastie lineárne** s každou správou používateľa a odpoveďou asistenta.
- **Tokeny v promptu tvoria hlavnú časť nákladov**, pretože celá história sa posiela znova v každom kole.
- Nakoniec konverzácia **prekoná kontextové okno** a model buď skráti obsah, alebo vyhodí chybu.

### Stratégií na spravovanie kontextu

| Stratégia | Ako funguje | Kompromis |
|---|---|---|
| **Skrátenie** | Odstráni najstaršie správy | Stratí sa skorší kontext |
| **Zhrnutie** | Zhrnie staršie správy do súhrnu | Niektoré detaily sa strácajú, ale hlavné body zostávajú |
| **Scratchpad / Externá pamäť** | Ukladá kľúčové fakty mimo konverzácie | Vyžaduje volania nástrojov, ale prežije akúkoľvek redukciu |

V tomto zápisníku kombinujeme **zhrnutie** s **nástrojom scratchpadu**, aby agent mohol udržiavať kontinuitu aj keď je historia konverzácie zhrnutá.


## Vytváranie agenta citlivého na kontext


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## Simulácia dlhej konverzácie

Poďme si prejsť viackolovú konverzáciu, aby sme videli, ako sa kontext kumuluje. Agent by si mal uchovávať kľúčové detaily (preferencie, rozpočet, dátumy cesty) počas jednotlivých kôl a preukázať kontinuitu.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Všimnite si, ako agent udržiava kontext z predchádzajúcich kôl — vie o Japonsku, sushi, chrámoch, fotografovaní, rozpočte 3000 dolárov, samostatnom cestovaní a časovom rámci apríla. V krátkom rozhovore to funguje dobre, ale keď sa rozhovor rozrastie, odosielanie celej histórie je nákladné.

Pokračujme v rozhovore s ďalšími kolami, aby sme videli kumuláciu kontextu:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Vzor zhrnutia kontextu

Ako sa konverzácia rozrastá, môžeme použiť **nástroj na zhrnutie** na zhrnutie nahromadeného kontextu do kompaktnej formy. Agent tento nástroj volá, aby zaznamenal kľúčové preferencie, takže aj keď sa staršie správy odstránia, základné informácie zostanú zachované.

Tento vzor je stavebným kameňom pre sofistikovanejšie znižovanie histórie:
1. Agent identifikuje kľúčové fakty z konverzácie
2. Volá nástroj na zhrnutie, aby ich uložil
3. Staršie správy je možné bezpečne odstrániť, pretože súhrn zachytáva to podstatné

Nižšie definujeme nástroj `summarize_preferences`, ktorý môže agent volať na zaznamenanie kompaktného zhrnutia toho, čo sa naučil.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Zhrnutie

V tejto lekcii ste sa naučili, ako spravovať kontext v dlhodobých konverzáciách agentov pomocou Microsoft Agent Framework:

### Kľúčové koncepty
- **Kontextové okná sú obmedzené** — každý token v histórii konverzácie stojí peniaze a započítava sa do limitu.
- **Nástroje na sumarizáciu** umožňujú agentovi zhrnúť nahromadený kontext do kompaktných súhrnov, čím znižujú používanie tokenov a zároveň zachovávajú podstatné informácie.
- **Poznámkové bloky agentov** poskytujú trvalú externú pamäť, ktorá prežije akékoľvek zmenšenie konverzácie.

### Čo ste vytvorili
- **Agent s uvedomovaním kontextu**, ktorý udržiava kontinuitu v rámci viackolových konverzácií
- **Nástroj na sumarizáciu** (`summarize_preferences`), ktorý zaznamenáva kľúčové údaje o používateľovi v kompaktnom formáte
- **Viackolovú konverzáciu**, ktorá demonštruje udržiavanie kontextu a spracovanie zmien

### Aplikácie v reálnom svete
- **Chatboty zákazníckej podpory**: Pamätajú si preferencie počas dlhých podporných relácií
- **Osobní asistenti**: Sledujú prebiehajúce projekty bez potreby opätovného vysvetľovania kontextu
- **Vzdelávací lektori**: Udržiavajú pokrok študenta naprieč viacerými interakciami

### Ďalšie kroky
- Implementovať plnohodnotný nástroj poznámkového bloku s uložením súborov
- Pridať automatické orezávanie histórie po sumarizácii
- Spojiť s vektorovými databázami pre sémantické vyhľadávanie v pamäti
- Stavať agentov, ktorí môžu obnovenie konverzácie o niekoľko dní neskôr s plným kontextom


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Zrieknutie sa zodpovednosti**:
Tento dokument bol preložený pomocou AI prekladateľskej služby [Co-op Translator](https://github.com/Azure/co-op-translator). Aj keď sa snažíme o presnosť, berte prosím na vedomie, že automatické preklady môžu obsahovať chyby alebo nepresnosti. Originálny dokument v jeho pôvodnom jazyku by sa mal považovať za autoritatívny zdroj. Pre kritické informácie sa odporúča profesionálny ľudský preklad. Nezodpovedáme za akékoľvek nedorozumenia alebo nesprávne interpretácie vyplývajúce z použitia tohto prekladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
